In [ ]:
%pip install --upgrade pandas

In [ ]:
import pandas as pd
print(f"Your new Pandas version is: {pd.__version__}")

In [ ]:
import json
import pandas as pd
import re
from collections import defaultdict
from azure.storage.blob import BlobServiceClient

ACCOUNT_URL = "https://aegisscholardata.blob.core.windows.net"
SAS_TOKEN = "?sv=2024-11-04&ss=bfqt&srt=sco&sp=rwdlacupiytfx&se=2026-05-30T08:31:15Z&st=2026-02-12T01:16:15Z&spr=https&sig=pU5C5a%2B%2BxE1zvMMv3vjUqjlJXC9dMgpsVyM0V%2FfuEIo%3D"
CONTAINER = "raw"
PREFIX = "dtic/works/" 

client = BlobServiceClient(account_url=ACCOUNT_URL, credential=SAS_TOKEN)
container = client.get_container_client(CONTAINER)

print("Connection initialized.")

In [ ]:
rows = []
print(f"Extracting data from {PREFIX}...")

blobs = container.list_blobs(name_starts_with=PREFIX)
processed_count = 0

for blob in blobs:
    if not blob.name.endswith('.json'):
        continue
        
    try:
        blob_client = container.get_blob_client(blob.name)
        data = blob_client.download_blob().readall()
        record = json.loads(data)
        
        paper_id = record.get('publication_id')
        authors = record.get('authors', [])
        num_authors = len(authors)
        
        topics = []
        for kw_group in record.get('keywords', []):
            for entity in kw_group.get('entities', []):
                name = entity.get('details', {}).get('name')
                if name: topics.append(name)
        
        for topic in topics:
            for auth in authors:
                rows.append({
                    'paper_id': paper_id,
                    'topic_full': topic,
                    'author_id': auth.get('researcher_id'),
                    'team_size': num_authors
                })
        
        processed_count += 1
        if processed_count % 500 == 0:
            print(f"Processed {processed_count} files...")

    except Exception:
        continue

dtic_df = pd.DataFrame(rows)
print(f"\nExtraction Complete! {len(dtic_df):,} relationships loaded.")

In [ ]:
import pandas as pd
import re

def parse_hierarchy(topic_string):
    if not isinstance(topic_string, str):
        return "Unknown", "Unknown"
    match = re.match(r'^(\d+)', topic_string)
    if not match:
        return "Unknown", "Unknown"
    code = match.group(1)
    cat = code[:2]
    sub = code[:4] if len(code) >= 4 else "N/A"
    return cat, sub

print("Processing hierarchy and stats...")

cats = []
subs = []
for t in dtic_df['topic_full']:
    c, s = parse_hierarchy(t)
    cats.append(c)
    subs.append(s)

dtic_df['category_code'] = cats
dtic_df['subcategory_code'] = subs

topic_stats = dtic_df.groupby(['category_code', 'subcategory_code', 'topic_full']).agg({
    'author_id': 'nunique',
    'team_size': 'mean',
    'paper_id': 'nunique'
}).reset_index()

topic_stats.columns = [
    'category_code', 'subcategory_code', 'topic_full', 
    'unique_authors', 'avg_authors_per_paper', 'total_papers'
]

topic_stats = topic_stats.sort_values(by=['category_code', 'unique_authors'], ascending=[True, False])
output_file = "DTIC_Topic_Analysis_Complete.csv"
topic_stats.to_csv(output_file, index=False)

print(f"\nSuccess! File saved to: {output_file}")
print(f"Total Rows: {len(topic_stats)}")

topic_stats.head(15)